# `rfq` — tender / request for quotation header

Unity Catalog: `ingestion_framework_test.bid_data_exploration.rfq`

Expected: one row per tender (the thing D-111808 was, in the sample bid-analyzer data on `full-feature-buildout`). Look for: a tender number / reference field, status, dates, description, and whether lots are modeled here or in another table.

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.rfq

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM ingestion_framework_test.bid_data_exploration.rfq

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq LIMIT 20

## Follow-up queries — after seeing the schema + first 20 rows

First pass findings (see full write-up in `databricks/FINDINGS.md`):
- **No lot column at all.** D-111808's 3 lots (SHBPRY/DRPRY/SMHPRY) have no home in this table — lots must live elsewhere (separate RFQ per lot? encoded in `quotationline`? need to check).
- **No explicit round column**, but `DISCOUNT_REVISION` and `POSTBID_DISCOUNT_CLOSEDATE` look like the real equivalent of "negotiation round" — worth confirming.
- **`ORGID`/`SITEID` show both `ADWEAORG`/`ADWEA` and `ADDCORG`/`ADDC`** — this table spans the whole legacy ADWEA group, not just ADDC. Needs filtering.
- **The unordered LIMIT 20 sample was all old (2002-2004, one 2018) records** — not representative of anything D-111808-like. Queries below target recent/relevant rows specifically.
- `DETAILBOQAVAILABLE` looks like the real equivalent of the sample data's `data_gap` flag (ELMEC-style bidders with no real BOQ).

### Is there a real analogue of D-111808 in here?

In [ ]:
%sql
SELECT RFQNUM, DESCRIPTION, STATUS, ORGID, SITEID, ENTERDATE, TOTALAWVALUE, DETAILBOQAVAILABLE
FROM ingestion_framework_test.bid_data_exploration.rfq
WHERE LOWER(DESCRIPTION) LIKE '%substation%' OR RFQNUM LIKE 'D-%'
ORDER BY ENTERDATE DESC
LIMIT 30

### What does a *recent* RFQ look like (not 2002-era)?

In [ ]:
%sql
SELECT RFQNUM, DESCRIPTION, STATUS, STAGE, ENTERDATE, TOTALAWVALUE, ORGID, DETAILBOQAVAILABLE
FROM ingestion_framework_test.bid_data_exploration.rfq
WHERE ENTERDATE >= '2024-01-01'
ORDER BY ENTERDATE DESC
LIMIT 30

### Which orgs/sites does this table actually cover?

In [ ]:
%sql
SELECT ORGID, SITEID, COUNT(*) AS n
FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY ORGID, SITEID
ORDER BY n DESC

### Status / stage / tender-status lifecycle values

In [ ]:
%sql
SELECT STATUS, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY STATUS ORDER BY n DESC

In [ ]:
%sql
SELECT STAGE, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY STAGE ORDER BY n DESC

In [ ]:
%sql
SELECT TENDERSTATUS, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY TENDERSTATUS ORDER BY n DESC

### Is `DETAILBOQAVAILABLE` really the data-gap flag?

In [ ]:
%sql
SELECT DETAILBOQAVAILABLE, COUNT(*) AS n
FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY DETAILBOQAVAILABLE ORDER BY n DESC

### Does `DISCOUNT_REVISION` behave like a negotiation-round counter?

In [ ]:
%sql
SELECT DISCOUNT_REVISION, COUNT(*) AS n
FROM ingestion_framework_test.bid_data_exploration.rfq
GROUP BY DISCOUNT_REVISION ORDER BY DISCOUNT_REVISION

**Observations (updated after first real run):**
- `RFQNUM` is the tender/RFQ number — but formats vary wildly across the table's history (`G1663`, `G-D2157`, `A11096521`, presumably `D-111808`-style too) — no single consistent pattern to parse by.
- No lot column anywhere in `rfq` — lots are not modeled at this level. Need to check `quotationline` for a lot indicator, or whether ADDC issues one RFQ per lot instead of one RFQ per tender.
- No explicit round column; `DISCOUNT_REVISION` / `POSTBID_DISCOUNT_CLOSEDATE` are the leading candidates for "which negotiation round" — pending the distribution query above.
- Table spans the whole ADWEA group (`ADWEA`, `ADDC`, and likely others) across ~20+ years (2002 through at least 2018 in the unordered sample) — any real query needs an `ORGID`/date filter, not a bare `LIMIT`.
- `DETAILBOQAVAILABLE` is a strong candidate for the real "data gap" flag (sample data's ELMEC-style bidders) — pending distribution check above.